# Upload scraped images to Cloudflare R2

Reads `vertexnetworking_products.csv`, downloads each product's image, uploads it to your Cloudflare R2 bucket, and writes a **new** file `vertexnetworking_products_r2.csv` — identical to the original except the `image` column now points to your R2-hosted URL instead of vertexnetworking.co.uk.

## Your Account ID is already filled in
`CLOUDFLARE_ACCOUNT_ID = "d42f7c5ed83f1403699b96fc13759c01"` — taken from your dashboard URL, already set in the code cell below.

## Still need to fill in 4 values
1. Go to your Cloudflare dashboard -> **R2 Object Storage** -> **Create bucket** (if you haven't already). Name it anything, e.g. `vertex-products`. Copy that name into `R2_BUCKET_NAME`.
2. In R2, click **Manage R2 API Tokens** -> **Create API Token**. Give it **Object Read & Write** permission. Copy the **Access Key ID** and **Secret Access Key** it shows you (only shown once) into `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY`.
3. Open your bucket -> **Settings** -> **Public Access** -> enable the **r2.dev subdomain** (easiest). Copy that public base URL (e.g. `https://pub-xxxxxxxxxxxx.r2.dev`) into `R2_PUBLIC_URL_BASE`.

In the code cell below, replace the remaining 4 placeholders:
```python
R2_ACCESS_KEY_ID = "PASTE_YOUR_ACCESS_KEY_ID"
R2_SECRET_ACCESS_KEY = "PASTE_YOUR_SECRET_ACCESS_KEY"
R2_BUCKET_NAME = "PASTE_YOUR_BUCKET_NAME"
R2_PUBLIC_URL_BASE = "PASTE_YOUR_PUBLIC_URL_BASE"
```

Then run the pip install cell, then the main cell. It's resumable and saves incrementally, same as the scraper.

In [1]:
!pip install requests boto3

In [ ]:
#!/usr/bin/env python3
"""
Takes the image URLs from your scraped vertexnetworking_products.csv,
downloads each image, uploads it to Cloudflare R2, and writes a NEW CSV
(all other columns unchanged) with the "image" column replaced by the
new R2-hosted URL.

Requirements:
    pip install requests boto3

Before running, fill in the CLOUDFLARE_* / R2_* settings below.

Output:
    vertexnetworking_products_r2.csv

Resumable: if you stop it and rerun, it skips product URLs already
present in the output CSV.
"""

import csv
import os
import re
import time
import hashlib
from urllib.parse import urlparse

import requests
import boto3
from botocore.config import Config

# ---------------------------------------------------------------------------
# CLOUDFLARE R2 SETTINGS — fill these in before running
# ---------------------------------------------------------------------------
# Where to find each value:
#   CLOUDFLARE_ACCOUNT_ID : Cloudflare dashboard -> right sidebar of any page
#                            (or R2 overview page URL contains it)
#   R2_ACCESS_KEY_ID /
#   R2_SECRET_ACCESS_KEY  : R2 -> "Manage R2 API Tokens" -> Create API Token
#                            (give it Object Read & Write permission)
#   R2_BUCKET_NAME         : the bucket you created in R2 (create one first
#                            if you haven't, e.g. "vertex-products")
#   R2_PUBLIC_URL_BASE     : R2 -> your bucket -> Settings -> Public Access
#                            -> enable the r2.dev subdomain (or connect a
#                            custom domain) and paste that base URL here,
#                            NO trailing slash, e.g.:
#                            "https://pub-xxxxxxxxxxxx.r2.dev"
CLOUDFLARE_ACCOUNT_ID = "d42f7c5ed83f1403699b96fc13759c01"
R2_ACCESS_KEY_ID = "PASTE_YOUR_ACCESS_KEY_ID"
R2_SECRET_ACCESS_KEY = "PASTE_YOUR_SECRET_ACCESS_KEY"
R2_BUCKET_NAME = "PASTE_YOUR_BUCKET_NAME"
R2_PUBLIC_URL_BASE = "PASTE_YOUR_PUBLIC_URL_BASE"

# ---------------------------------------------------------------------------
INPUT_CSV = "vertexnetworking_products.csv"
OUTPUT_CSV = "vertexnetworking_products_r2.csv"
REQUEST_DELAY = 0.3     # seconds between downloads, be polite to the source site
TIMEOUT = 20
PROGRESS_EVERY = 25
KEY_PREFIX = "products/"   # folder-like prefix inside the R2 bucket

CONTENT_TYPES = {
    ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    ".png": "image/png", ".webp": "image/webp",
    ".gif": "image/gif", ".bmp": "image/bmp",
}

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
})

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    config=Config(signature_version="s3v4"),
    region_name="auto",
)


def safe_filename(url, part_number=""):
    """Build a stable, filesystem/URL-safe filename for the uploaded image."""
    parsed = urlparse(url)
    ext = os.path.splitext(parsed.path)[1].lower()
    if ext not in CONTENT_TYPES:
        ext = ".jpg"
    base = (part_number or "").strip()
    if not base:
        base = hashlib.md5(url.encode("utf-8")).hexdigest()[:12]
    base = re.sub(r"[^A-Za-z0-9_\-]", "_", base)
    return f"{base}{ext}"


def download_image(url):
    try:
        resp = session.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        return resp.content
    except requests.RequestException as e:
        print(f"  [warn] failed to download {url}: {e}")
        return None


def upload_to_r2(content, key, content_type):
    try:
        s3.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=key,
            Body=content,
            ContentType=content_type,
        )
        return f"{R2_PUBLIC_URL_BASE.rstrip('/')}/{key}"
    except Exception as e:
        print(f"  [warn] failed to upload {key} to R2: {e}")
        return None


def load_already_done():
    """Rows already written to the output CSV, keyed by product url."""
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("url"):
                    done.add(row["url"])
    return done


def main():
    if CLOUDFLARE_ACCOUNT_ID.startswith("PASTE_"):
        raise SystemExit(
            "Fill in the CLOUDFLARE_ACCOUNT_ID / R2_* settings near the top "
            "of this cell before running."
        )
    if not os.path.exists(INPUT_CSV):
        raise SystemExit(f"Can't find {INPUT_CSV} — put this notebook in the same folder.")

    already_done = load_already_done()
    if already_done:
        print(f"Resuming — {len(already_done)} rows already in {OUTPUT_CSV}, will skip those.")

    with open(INPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    total = len(rows)
    print(f"Loaded {total} rows from {INPUT_CSV}.")

    file_exists = os.path.exists(OUTPUT_CSV)
    out_f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(out_f, fieldnames=fieldnames)
    if not file_exists:
        writer.writeheader()
        out_f.flush()

    processed = len(already_done)

    try:
        for row in rows:
            product_url = row.get("url", "")
            if product_url and product_url in already_done:
                continue

            new_row = dict(row)
            image_url = (row.get("image") or "").strip()

            if image_url:
                content = download_image(image_url)
                if content:
                    ext = os.path.splitext(urlparse(image_url).path)[1].lower()
                    content_type = CONTENT_TYPES.get(ext, "image/jpeg")
                    key = KEY_PREFIX + safe_filename(image_url, row.get("part_number", ""))
                    new_url = upload_to_r2(content, key, content_type)
                    if new_url:
                        new_row["image"] = new_url
                    time.sleep(REQUEST_DELAY)

            writer.writerow(new_row)
            out_f.flush()
            processed += 1

            if processed % PROGRESS_EVERY == 0:
                print(f"  >>> progress: {processed}/{total} rows processed")

    except KeyboardInterrupt:
        print("\n[stopped by user] Progress so far is safely saved.")
    finally:
        out_f.close()

    print(f"\nDone. {processed}/{total} rows written to {OUTPUT_CSV}")
    print("(rerun this cell anytime to resume/finish remaining rows)")


if __name__ == "__main__":
    main()


Loaded 83146 rows from vertexnetworking_products.csv.
  [warn] failed to upload products/M0T01A.webp to R2: An error occurred (InvalidArgument) when calling the PutObject operation: Credential access key has length 24, should be 32
  [warn] failed to upload products/64116B4.webp to R2: An error occurred (InvalidArgument) when calling the PutObject operation: Credential access key has length 24, should be 32
  [warn] failed to upload products/M0T00A.webp to R2: An error occurred (InvalidArgument) when calling the PutObject operation: Credential access key has length 24, should be 32
  [warn] failed to upload products/AV451A.webp to R2: An error occurred (InvalidArgument) when calling the PutObject operation: Credential access key has length 24, should be 32
  [warn] failed to upload products/64116B2.webp to R2: An error occurred (InvalidArgument) when calling the PutObject operation: Credential access key has length 24, should be 32
  [warn] failed to upload products/8TTVC.webp to R2: A